In [1]:
import os

In [2]:
%pwd

'/workspaces/End-to-End-MLOps-Bootcamp-Build-Deploy-and-Automate-ML-with-Data-Science-Projects/myFirstNewProject/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/workspaces/End-to-End-MLOps-Bootcamp-Build-Deploy-and-Automate-ML-with-Data-Science-Projects/myFirstNewProject'

In [5]:
import pandas as pd

data = pd.read_csv("artifacts/data_ingestion/winequality-red.csv")
data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [9]:
## Step-1: Creating entity classes for Data Transformation Configurations
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataTransformationConfig:
    root_dir: Path
    data_path: Path

## Step-2: Update the configuration manager to read data transformation config from YAML file
from src.my_first_end_to_end_project.constants import *
from src.my_first_end_to_end_project.utils.common_utils import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
            self,
            config_filepath=CONFIG_FILE_PATH,
            params_filepath=PARAMS_FILE_PATH,
            SCHEMA_FILE_PATH = SCHEMA_FILE_PATH
            ):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(SCHEMA_FILE_PATH)
        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self)->DataTransformationConfig:
        config = self.config.data_transformation
        create_directories([config.root_dir])
        data_transformation_config = DataTransformationConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path)
        )
        return data_transformation_config
    

## Step-3: Update components to use the configuration manager to get data transformation config
import os
from src.my_first_end_to_end_project.logger import logger
from sklearn.model_selection import train_test_split
import pandas as pd  

class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        
        
    ## We can utilize all the data transformation techniques here like PCA, Scaling, encoding and other here
    def split_train_test(self):
        data = pd.read_csv(self.config.data_path)
        train, test = train_test_split(data)

        # save the train and test data
        train.to_csv(os.path.join(self.config.root_dir, "train.csv"), index=False)
        test.to_csv(os.path.join(self.config.root_dir, "test.csv"), index=False)

        logger.info(f"Train test split completed. ")
        logger.info(f"Train data shape : {train.shape}")
        logger.info(f"Test data shape : {test.shape}")

        print(train.shape)
        print(test.shape)  


In [10]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.split_train_test()
except Exception as e:
    logger.exception(e)
    raise e

2026-01-06 17:17:44 - INFO - common_utils - yaml file: config/config.yaml is loaded successfully 🥳
2026-01-06 17:17:44 - INFO - common_utils - yaml file: params.yaml is loaded successfully 🥳
2026-01-06 17:17:44 - INFO - common_utils - yaml file: schema.yaml is loaded successfully 🥳
2026-01-06 17:17:44 - INFO - common_utils - Created directory Successfuly at: artifacts 🥳
2026-01-06 17:17:44 - INFO - common_utils - Created directory Successfuly at: artifacts/data_transformation 🥳
2026-01-06 17:17:44 - INFO - 2925814296 - Train test split completed. 
2026-01-06 17:17:44 - INFO - 2925814296 - Train data shape : (1199, 12)
2026-01-06 17:17:44 - INFO - 2925814296 - Test data shape : (400, 12)
(1199, 12)
(400, 12)


In [ ]:
# Step-4: Update pipeline to include data transformation step
# Step-5: Update main.py